In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

c:\Users\jaesc2\GitHub\skforecast
0.22.0


Yes, there are a few significantly faster alternatives to the default `statsmodels` ACF implementation, especially if you are dealing with large time series. The slowness you are experiencing is likely because the default method calculates correlation in the time domain, which has a time complexity of $O(N^2)$. 

By switching to Fast Fourier Transform (FFT) based methods, you reduce the time complexity to $O(N \log N)$, which is orders of magnitude faster for long sequences.

Here are the best and fastest alternatives, ranked from easiest to implement to most extreme.

### 1. The Quick Fix: Enable FFT in Statsmodels (Fast)
You might not actually need to leave `statsmodels`. The `acf` function has a built-in `fft` argument that defaults to `False`. Simply flipping this to `True` usually solves the performance bottleneck immediately.

```python
from statsmodels.tsa.stattools import acf

# Set fft=True for a massive speedup on large arrays
lags = acf(your_time_series, nlags=100, fft=True)
```

### 2. The SciPy FFT Approach (Faster)
If you want to strip away the overhead of `statsmodels` entirely, you can use `scipy.signal.correlate`. By explicitly setting `method='fft'`, you force SciPy to use the fastest possible calculation route. 

Because `correlate` returns the full cross-correlation (both negative and positive lags), you just need to slice the second half of the array and normalize it.

```python
import numpy as np
from scipy import signal

def fast_acf_scipy(x, nlags):
    # Mean center the data
    x_centered = x - np.mean(x)
    
    # Compute full autocorrelation using FFT
    autocorr_full = signal.correlate(x_centered, x_centered, mode='full', method='fft')
    
    # Extract the second half (lags 0 to N)
    autocorr = autocorr_full[len(autocorr_full)//2:]
    
    # Normalize by the variance (lag 0)
    autocorr /= autocorr[0]
    
    # Return only the requested number of lags
    return autocorr[:nlags+1]

lags = fast_acf_scipy(your_time_series, nlags=100)
```

### 3. Pure NumPy FFT (Fastest CPU Method)
If you don't want to rely on SciPy or `statsmodels` at all, you can manually calculate the autocorrelation using the Convolution Theorem via NumPy's FFT module. This is essentially what SciPy is doing under the hood, but writing it natively cuts out all library overhead.

```python
import numpy as np

def fast_acf_numpy(x, nlags):
    n = len(x)
    x_centered = x - np.mean(x)
    
    # Pad to next power of 2 for optimal FFT speed
    n_fft = 2 ** int(np.ceil(np.log2(2 * n - 1)))
    
    # Compute FFT
    fft_x = np.fft.fft(x_centered, n=n_fft)
    
    # Multiply by complex conjugate to get the power spectrum
    power_spectrum = fft_x * np.conjugate(fft_x)
    
    # Inverse FFT to get autocorrelation in time domain
    autocorr = np.fft.ifft(power_spectrum).real
    
    # Slice, normalize, and return requested lags
    autocorr = autocorr[:nlags+1]
    return autocorr / autocorr[0]

lags = fast_acf_numpy(your_time_series, nlags=100)
```


In [ ]:
import numpy as np
import time
from statsmodels.tsa.stattools import acf
from scipy import signal

# ==========================================
# 1. Define the Custom FFT ACF Functions
# ==========================================

def fast_acf_scipy(x, nlags):
    """Calculates ACF using SciPy's FFT correlation."""
    x_centered = x - np.mean(x)
    # Compute full autocorrelation using FFT
    autocorr_full = signal.correlate(x_centered, x_centered, mode='full', method='fft')
    # Extract the second half (lags 0 to N)
    autocorr = autocorr_full[len(autocorr_full)//2:]
    # Normalize and return requested lags
    return autocorr[:nlags+1] / autocorr[0]

def fast_acf_numpy(x, nlags):
    """Calculates ACF manually using Pure NumPy FFT."""
    n = len(x)
    x_centered = x - np.mean(x)
    # Pad to next power of 2 for optimal FFT speed
    n_fft = 2 ** int(np.ceil(np.log2(2 * n - 1)))
    # Compute FFT
    fft_x = np.fft.fft(x_centered, n=n_fft)
    # Power spectrum (Convolution Theorem)
    power_spectrum = fft_x * np.conjugate(fft_x)
    # Inverse FFT to time domain
    autocorr = np.fft.ifft(power_spectrum).real
    # Normalize and return requested lags
    autocorr = autocorr[:nlags+1]
    return autocorr / autocorr[0]

# ==========================================
# 2. Setup Benchmark Parameters
# ==========================================

# Create a large synthetic time series (e.g., random walk / AR process)
np.random.seed(42)
N = 10_000          # 10,000 data points
nlags = 500          # Calculate first 500 lags
x = np.cumsum(np.random.randn(N)) # Cumulative sum creates a random walk

print(f"Dataset Size: {N:,} points")
print(f"Lags Calculated: {nlags}\n")

# Dictionary of methods to test
methods = {
    "Statsmodels (fft=False)": lambda: acf(x, nlags=nlags, fft=False),
    "Statsmodels (fft=True) ": lambda: acf(x, nlags=nlags, fft=True),
    "SciPy FFT Method       ": lambda: fast_acf_scipy(x, nlags),
    "Pure NumPy FFT Method  ": lambda: fast_acf_numpy(x, nlags)
}

# ==========================================
# 3. Run Benchmark and Compare Values
# ==========================================

results = {}
baseline_values = None
runs = 10 # Number of times to run each method for average speed

print(f"{'Method':<25} | {'Avg Time (ms)':<15} | {'Max Diff vs Baseline'}")
print("-" * 65)

for name, func in methods.items():
    # 1. Get values and check accuracy
    vals = func()
    
    if baseline_values is None:
        baseline_values = vals
        max_diff = 0.0
    else:
        # Check difference against the baseline (Statsmodels fft=False)
        max_diff = np.max(np.abs(vals - baseline_values))
        
    # 2. Time the performance
    start_time = time.perf_counter()
    for _ in range(runs):
        func()
    end_time = time.perf_counter()
    
    avg_time_ms = ((end_time - start_time) / runs) * 1000
    
    # 3. Print row
    # Using scientific notation for difference to show precise floating-point matches
    print(f"{name:<25} | {avg_time_ms:>13.2f} ms | {max_diff:>10.2e}")

In [ ]:
import numpy as np
from scipy import signal
import scipy.stats as stats

def _calc_confint(acf_vals, n, alpha):
    """
    Helper function to calculate Bartlett's confidence intervals.
    """
    # 1. Look up the z-score for the given alpha
    z = stats.norm.ppf(1 - alpha / 2.0)
    
    # 2. Initialize variance array (1/N for all lags initially)
    varacf = np.ones(len(acf_vals)) / n
    varacf[0] = 0  # Lag 0 is exactly 1.0, so it has 0 variance
    
    # 3. Apply Bartlett's formula for lags 2 and beyond
    if len(acf_vals) > 2:
        # We only cumsum the squared lags from lag 1 to k-1
        varacf[2:] *= 1 + 2 * np.cumsum(acf_vals[1:-1]**2)
        
    # 4. Calculate the margin of error
    interval = z * np.sqrt(varacf)
    
    # 5. Return as a 2D array: [lower_bound, upper_bound]
    return np.column_stack((acf_vals - interval, acf_vals + interval))


def fast_acf_scipy(x, nlags, alpha=None):
    """Calculates ACF using SciPy's FFT correlation, with optional CI."""
    n = len(x)
    x_centered = x - np.mean(x)
    
    # Compute full autocorrelation using FFT
    autocorr_full = signal.correlate(x_centered, x_centered, mode='full', method='fft')
    autocorr = autocorr_full[len(autocorr_full)//2:]
    
    # Normalize and slice
    acf_vals = autocorr[:nlags+1] / autocorr[0]
    
    if alpha is not None:
        confint = _calc_confint(acf_vals, n, alpha)
        return acf_vals, confint
    return acf_vals


def fast_acf_numpy(x, nlags, alpha=None):
    """Calculates ACF manually using Pure NumPy FFT, with optional CI."""
    n = len(x)
    x_centered = x - np.mean(x)
    
    # Pad to next power of 2 for optimal FFT speed
    n_fft = 2 ** int(np.ceil(np.log2(2 * n - 1)))
    
    # Compute FFT and power spectrum
    fft_x = np.fft.fft(x_centered, n=n_fft)
    power_spectrum = fft_x * np.conjugate(fft_x)
    
    # Inverse FFT to time domain
    autocorr = np.fft.ifft(power_spectrum).real
    
    # Normalize and slice
    acf_vals = autocorr[:nlags+1] / autocorr[0]
    
    if alpha is not None:
        confint = _calc_confint(acf_vals, n, alpha)
        return acf_vals, confint
    return acf_vals

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import levinson_durbin

def fast_pacf_numpy(x, nlags):
    # ---------------------------------------------------------
    # Step 1: Calculate ACF using our ultra-fast NumPy FFT method
    # ---------------------------------------------------------
    n = len(x)
    x_centered = x - np.mean(x)
    n_fft = 2 ** int(np.ceil(np.log2(2 * n - 1)))
    
    fft_x = np.fft.fft(x_centered, n=n_fft)
    power_spectrum = fft_x * np.conjugate(fft_x)
    
    autocorr = np.fft.ifft(power_spectrum).real
    acf_vals = autocorr[:nlags+1] / autocorr[0]
    
    # ---------------------------------------------------------
    # Step 2: Feed the fast ACF into the Levinson-Durbin solver
    # ---------------------------------------------------------
    # isacov=True tells the function we are feeding it correlations, not raw data.
    # The function returns a tuple; the PACF values are the 3rd element at index [2].
    pacf_vals = levinson_durbin(acf_vals, nlags=nlags, isacov=True)[2]
    
    return pacf_vals

# Run it
pacf_vals = fast_pacf_numpy(your_time_series, nlags=100)